In [1]:
from winnow_fcns import *
#first to siffting
#run error orrectin protocal on that
#do naive agreement

In [ ]:
# Initalizations

# 1. First, you'd likely have your BitBuffer ready (assuming BitBuffer is defined elsewhere)
my_sifted_key = BitBuffer("11010101") 

# 2. Choose a seed for the random permutation (shuffling)
seed_value = 42

# 3. Initialize the Winnow instance
winnow_instance = Winnow(raw_key=my_sifted_key, perm_seed=seed_value)



In [ ]:
def create_simulated_key(rng, ber, std, N):
    err = rng.normal(loc=ber, scale=std, N=N)
    mask = err > ber
    cooked_key =  key + mask

    

1) Divide key into small blocks. The key is the bitstring we obtained from the matlab script

In [ ]:
def divide_key(key, block_size):
    

2) For each block, calculate a hamming code syndrome

3) See if syndromes match
4) If syndromes don't match use syndrome table to determine correction to apply

In [ ]:
get_syndrome(8)

5) Alice sends syndrome to Bob (EXPORT TO MATLAB W CSV)
6) Calculate how much info has been given to Eve. after a certain threshold throw out the key (restart with a new key?)

7) apply hashing algorithm to the key to handle eve

In [3]:
import numpy as np
# Parameters
N_BITS = 128       # Total length of the sifted key
BER = 0.05         # 5% error rate (Winnow works best when errors are sparse)
BLOCK_SIZE = 8     # Standard for first pass
SEED = 42          # Must be identical for Alice and Bob - a seed for the random permutation (shuffling)

In [7]:
class MockBitBuffer:
    def __init__(self, bits):
        self.bits = list(bits)
        self.seed = None
    def get_length(self): return len(self.bits)
    def get_bit(self, i): return self.bits[i]
    def set_bit(self, i): self.bits[i] = 1
    def clear_bit(self, i): self.bits[i] = 0
    def flip_bit(self, i): self.bits[i] = 1 - self.bits[i]
    def set_seed(self, s): self.seed = s
    def permute_buffer(self): pass # Simplified for test

In [5]:
def simulate_error(key, rng, ber, N):
    mask = (rng.random(size=N) < ber).astype(int)
    return key ^ mask

In [16]:
# Generate Alice's Key
rng_stream = np.random.default_rng(seed=42)
rng_key = np.random.default_rng(seed=180)

# 1. Generate Alice's raw bits
alice_raw = rng_key.integers(low=0, high=2, size=N_BITS)

# 2. Generate Bob's raw bits 
bob_raw = simulate_error(alice_raw, rng_stream, ber=BER, N=N_BITS)

# 3. Wrap them for the Winnow Class
alice_key = MockBitBuffer(alice_raw)
bob_key   = MockBitBuffer(bob_raw)

print(f"Initial errors: {np.sum(alice_raw != bob_raw)}")

# --- Initialize Winnow ---
alice_winnow = Winnow(raw_key=alice_key, perm_seed=42)
bob_winnow = Winnow(raw_key=bob_key, perm_seed=42)

alice_winnow.first_pass()
bob_winnow.first_pass()

Initial errors: 5


0

### Multiple Passes

In [10]:
import numpy as np

class MockBitBuffer:
    def __init__(self, bits):
        # Convert to a numpy array for easy manipulation and shuffling
        self.bits = np.array(bits, dtype=int)
        self.seed = None
        
    def get_length(self): 
        """Returns current length of the buffer."""
        return len(self.bits)
    
    def get_bit(self, i): 
        """Returns the bit at index i."""
        return self.bits[i]
    
    def set_bit(self, i, val=1): 
        """Sets the bit at index i to a specific value (default 1)."""
        self.bits[i] = val
        
    def clear_bit(self, i): 
        """Sets the bit at index i to 0."""
        self.bits[i] = 0
        
    def flip_bit(self, i): 
        """Flips the bit at index i (0 to 1 or 1 to 0)."""
        self.bits[i] = 1 - self.bits[i]
        
    def set_seed(self, s): 
        """Sets the seed for the random permutation."""
        self.seed = s
        
    def get_seed(self): 
        """Returns the current seed."""
        return self.seed
    
    def set_length(self, n): 
        """Physically truncates the buffer to length n. Used in discarding bits."""
        self.bits = self.bits[:n]

    def get_parity(self, start, end):
        """Calculates the parity of a slice [start, end]. 1 if odd, 0 if even."""
        # sum() % 2 is a fast way to get binary parity
        return np.sum(self.bits[start:end+1]) % 2
    
    def permute_buffer(self):
        """
        Shuffles the bits based on the seed. 
        Crucial for multi-pass Winnow to spread out error clusters.
        """
        if self.seed is None:
            raise ValueError("Seed must be set before permutation!")
        
        # We use a dedicated Random Number Generator (RNG) with our seed
        # to ensure Alice and Bob shuffle their keys in the EXACT same way.
        rng = np.random.default_rng(self.seed)
        rng.shuffle(self.bits)

In [11]:
def simulate_error(key, rng, ber, N):
    mask = (rng.random(size=N) < ber).astype(int)
    return key ^ mask

In [12]:
# --- 1. RESET AND INITIALIZE WITH LARGER N ---
N = 1024           # Use 1024 to ensure we have enough bits after discarding
BER = 0.05
SHARED_SEED = 42

rng_key = np.random.default_rng(seed=180)
rng_noise = np.random.default_rng(seed=123)

alice_raw = rng_key.integers(0, 2, N)
bob_raw = simulate_error(alice_raw, rng_noise, BER, N)

alice_key = MockBitBuffer(alice_raw)
bob_key = MockBitBuffer(bob_raw)

alice_winnow = Winnow(alice_key, perm_seed=SHARED_SEED)
bob_winnow = Winnow(bob_key, perm_seed=SHARED_SEED)

# Define the schedule
alice_winnow._block_size_schedule = [2, 1, 1, 0, 0, 0, 0, 0]
bob_winnow._block_size_schedule = [2, 1, 1, 0, 0, 0, 0, 0]

# --- 2. RUN THE PASSES ---
alice_winnow.first_pass()
bob_winnow.first_pass()

pass_count = 1
while True:
    # RE-CALCULATE num_of_blocks right here to be safe
    current_num_blocks = alice_winnow._key_string.get_length() // alice_winnow._block_size
    alice_winnow._num_of_blocks = current_num_blocks
    bob_winnow._num_of_blocks = current_num_blocks
    
    print(f"Pass {pass_count}: Blocks={current_num_blocks}, BlockSize={alice_winnow._block_size}, KeyLen={alice_key.get_length()}")
    
    # 1. Collect syndromes
    syndromes = []
    for i in range(current_num_blocks):
        syndromes.append(alice_winnow.get_syndrome(i))
    
    # 2. Fix Bob's key
    for i in range(current_num_blocks):
        bob_winnow.fix_with_syndrome(i, syndromes[i])
    
    # 3. Discard bits (Privacy step)
    active_blocks = list(range(current_num_blocks))
    alice_winnow.discard_syndrome_bits(active_blocks)
    bob_winnow.discard_syndrome_bits(active_blocks)
    
    # 4. Advance to next pass
    # next_pass returns -1 when the schedule [2, 1, 1...] is empty
    status = alice_winnow.next_pass(permute_bits=True)
    bob_winnow.next_pass(permute_bits=True)
    
    if status == -1:
        print("Done!")
        break
    pass_count += 1

# Final verification
if np.array_equal(alice_key.bits, bob_key.bits):
    print(f"✅ Success! Final key length: {alice_key.get_length()}")
else:
    print("❌ Keys do not match.")

Pass 1: Blocks=128, BlockSize=8, KeyLen=1024
Pass 2: Blocks=64, BlockSize=8, KeyLen=512
Pass 3: Blocks=32, BlockSize=8, KeyLen=256
Pass 4: Blocks=8, BlockSize=16, KeyLen=128
Pass 5: Blocks=2, BlockSize=32, KeyLen=88
Time to terminate.
Time to terminate.
Done!
✅ Success! Final key length: 76


In [ ]:
# --- 1. SETUP ---
N = 64
BER = 0.10  # 10% error rate (approx 6-7 errors)
SHARED_SEED = 42

rng_key = np.random.default_rng(seed=180)
rng_noise = np.random.default_rng(seed=123)

# Create Alice's key and a noisy Bob's key
alice_bits = rng_key.integers(0, 2, N)
bob_bits = simulate_error(alice_bits, rng_noise, BER, N)

# Print initial state
print(f"Original Alice: {''.join(map(str, alice_bits))}")
print(f"Original Bob:   {''.join(map(str, bob_bits))}")
print(f"Initial Diff:   {np.sum(alice_bits != bob_bits)} bits\n")

# Wrap into Winnow
alice_key = MockBitBuffer(alice_bits)
bob_key = MockBitBuffer(bob_bits)
alice_winnow = Winnow(alice_key, perm_seed=SHARED_SEED)
bob_winnow = Winnow(bob_key, perm_seed=SHARED_SEED)

# Define schedule for a short test
alice_winnow._block_size_schedule = [2, 1, 0, 0, 0, 0, 0, 0]
bob_winnow._block_size_schedule = [2, 1, 0, 0, 0, 0, 0, 0]

# --- 2. EXECUTION ---
alice_winnow.first_pass()
bob_winnow.first_pass()

pass_count = 1
while True:
    current_num_blocks = alice_winnow._key_string.get_length() // alice_winnow._block_size
    
    # Alice sends syndromes
    syndromes = [alice_winnow.get_syndrome(i) for i in range(current_num_blocks)]
    
    # Bob fixes
    for i in range(current_num_blocks):
        bob_winnow.fix_with_syndrome(i, syndromes[i])
    
    # Winnowing (Discard bits)
    active_blocks = list(range(current_num_blocks))
    alice_winnow.discard_syndrome_bits(active_blocks)
    bob_winnow.discard_syndrome_bits(active_blocks)
    
    print(f"Pass {pass_count} Complete. Current Key Length: {alice_key.get_length()}")
    
    # Shuffle for next pass
    status = alice_winnow.next_pass(permute_bits=True)
    bob_winnow.next_pass(permute_bits=True)
    
    if status == -1:
        break
    pass_count += 1

# --- 3. FINAL RESULTS ---
final_a = "".join(map(str, alice_key.bits))
final_b = "".join(map(str, bob_key.bits))

print(f"\nFinal Alice: {final_a}")
print(f"Final Bob:   {final_b}")

if final_a == final_b:
    print("\n SUCCESS! The bit strings match perfectly.")
else:
    print("\nr FAILURE: There are still differences.")

Original Alice: 0111011000110011011001000001111000001010110100100101010010001010
Original Bob:   0011011000110011011001000101111000001010010100010101110010001010
Initial Diff:   6 bits

Pass 1 Complete. Current Key Length: 32
Pass 2 Complete. Current Key Length: 16
Pass 3 Complete. Current Key Length: 8
Pass 4 Complete. Current Key Length: 8
Time to terminate.
Time to terminate.

Final Alice: 10011000
Final Bob:   10011000

✅ SUCCESS! The bit strings match perfectly.
